# Tools 자체 AI 학습 노트 (Kaggle용)

Frenchfries777 Tools의 자체 AI를 학습시키는 노트예요. Colab 노트와 같은 학습이고, 토큰과 저장 방식만 Kaggle에 맞췄어요.

처음 한 번만 해 주세요.
1. Kaggle에 로그인하고 휴대폰 번호 인증을 해요 (GPU와 인터넷을 쓰려면 필요해요).
2. 이 노트를 **File → Import Notebook** 으로 열어요 (GitHub 주소 또는 파일).
3. 오른쪽 **Session options** 에서 **Accelerator = GPU T4 x2** (또는 P100), **Internet = On** 으로 해요.
4. 메뉴 **Add-ons → Secrets** 에서 이름 `HF_TOKEN`, 값은 Hugging Face **Write** 토큰으로 추가하고 이 노트에 체크해 둬요 (토큰은 노트 칸에 쓰지 말아요).

실행
5. 아래 **설정** 칸에서 `MODEL` 과 `CONTINUE`, `ROUND` 를 Colab과 똑같이 골라요. (타리우스 = basic, 렉투스 = smart)
6. 오른쪽 위 **Save Version → Save & Run All (Commit)** 을 눌러요. 화면을 꺼도 계속 돌고, 한 번에 최대 약 9시간이에요. 끝나면 Hugging Face에 비공개로 올려요.

그냥 위에서부터 ▶ 를 눌러도 되지만 그러면 탭을 열어 둬야 해요. GPU는 일주일에 약 30시간까지 무료예요. 렉투스는 약 4~4.5시간이라 한 번에 끝나요 (잰 값은 아니에요).

중간에 끊기면 그 버전의 Output을 새 노트의 Input으로 추가하고 다시 실행하면 이어서 해요.
끝나면 "올림 끝"이 나와요. Claude에게 알려 주세요.


In [ ]:
#@title 설정
MODEL = "basic"  #@param ["basic", "smart"]
HF_TOKEN = ""  #@param {type:"string"}
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # unsloth uses one GPU (T4 x2: the second one stays unused)
if not HF_TOKEN:   # Kaggle: the token comes from Add-ons -> Secrets (name HF_TOKEN), never typed into the notebook
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Secrets에서 HF_TOKEN을 못 읽었어요:", e)
RUN_EVAL = True  #@param {type:"boolean"}
CONTINUE = False  #@param {type:"boolean"}
ROUND = 3  #@param [2, 3]

REPO_RAW = "https://raw.githubusercontent.com/frenchfries777mc/tools/main/train/data/"
CFG = {
  "basic": dict(base="unsloth/Qwen3-1.7B", size="1.7B", name="tarius", data=["basic.jsonl", "round2-basic.jsonl"],           epochs=1, r=8,  lr=2e-4),
  "smart": dict(base="unsloth/Qwen3-4B",   size="4B",   name="lectus", data=["full.jsonl", "round2-smart.jsonl", "round3-smart.jsonl"],           epochs=2, r=16, lr=1.5e-4),
}[MODEL]
if CONTINUE:   # keep training the model already on Hugging Face (only the second-round data), gently
    CFG.update(base=f"Frenchfries777/FF-Tools-{CFG['size']}-merged", data=[f"round{ROUND if MODEL == 'basic' else 2}-{MODEL}.jsonl"], epochs=1, lr=CFG["lr"] / 4)
MAX_LEN = 4096
print("학습할 모델:", "기본" if MODEL == "basic" else "똑똑함", CFG)
assert HF_TOKEN.startswith("hf_"), "Hugging Face 토큰(hf_로 시작)을 넣어 주세요"

## 1. 준비 (프로그램 설치)

In [ ]:
%%capture
!pip install -q unsloth

In [ ]:
import json, re, math, random, time, urllib.request, os, subprocess
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 T4 GPU로 바꿔 주세요)")
assert torch.cuda.is_available()

def load(name):
    with urllib.request.urlopen(REPO_RAW + name) as r:
        return [json.loads(l) for l in r.read().decode("utf-8").splitlines() if l.strip()]
train_rows = [r for f in CFG["data"] for r in load(f)]; test_rows = load("test.jsonl")

# What the developer taught the AI on the site's 가르치기 page (public read; only the developer can write)
ASK_SYS = 'You help someone with their spreadsheet. Answer the question about the table clearly and briefly in plain text (no markdown tables, no code fences). Use the language the question is written in.'
WHO = {"tarius": 'You are FF777 1.0 tarius, the built-in AI of the Frenchfries777 Tools site, made by Frenchfries777 from Qwen3 1.7B. Your name is the end of the Latin word elementarius ("basic").', "lectus": 'You are FF777 1.0 lectus, the built-in AI of the Frenchfries777 Tools site, made by Frenchfries777 from Qwen3 4B. Your name is the end of the Latin word intellectus ("understanding").'}[CFG["name"]]
facts = []
try:
    req = urllib.request.Request("https://kebsbuvuutilitwlvxzv.supabase.co/rest/v1/ff_ai_facts?select=model,question,fact&order=id",
                                 headers={"apikey": "sb_publishable_4Egu0zm3p6cxWGvP_-GlIw_mlCpO1Nf"})
    facts = [f for f in json.load(urllib.request.urlopen(req)) if f["model"] in ("all", CFG["name"]) and f["fact"]]
except Exception as e:
    print("가르친 내용을 불러오지 못했어요 (없이 계속해요):", e)
lines = [("- " + (f["question"] + " -> " if f["question"] else "") + f["fact"]) for f in facts]
with_facts = ASK_SYS + "\n\n" + WHO + ("\nFacts you know (use them when they are relevant):\n" + "\n".join(lines[-30:])[-2000:] if lines else "")
for f in facts:
    if not f["question"]: continue
    ask = "(No table selected.)\n\nQuestion: " + f["question"]
    for sysmsg, times in ((with_facts, 1), (ASK_SYS + "\n\n" + WHO, 2)):   # with the facts in the prompt, and from memory
        for _ in range(times):
            train_rows.append({"task": "taught", "messages": [{"role": "system", "content": sysmsg}, {"role": "user", "content": ask}, {"role": "assistant", "content": f["fact"]}]})
print("가르친 내용", len(facts), "개 (질문이 있는 것만 학습 자료로 들어가요)")
from collections import Counter
print("학습 자료", len(train_rows), dict(Counter(r["task"] for r in train_rows)))
print("시험", len(test_rows), dict(Counter(r["task"] for r in test_rows)))

## 2. 바탕 모델 불러오기

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(CFG["base"], max_seq_length=MAX_LEN, load_in_4bit=True, token=HF_TOKEN)

def prompt_text(msgs):   # what the site sends (thinking off), up to where the answer starts
    return tokenizer.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)
def full_text(msgs):
    return tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)
print(prompt_text(train_rows[0]["messages"])[-300:])

## 3. 시험 채점 (학습 전)

시험 문제 일부를 풀게 해서 점수를 재요. 학습이 끝난 뒤 같은 문제로 다시 재서 비교해요.

In [ ]:
CAT = None
def catalog():
    global CAT
    if CAT is None:
        with urllib.request.urlopen(REPO_RAW.replace("/data/", "/ref/") + "catalog.txt") as r:
            CAT = set(l.split(" ")[0] for l in r.read().decode("utf-8").splitlines() if l.strip())
    return CAT

def chrf(hyp, ref, n=4):   # character n-gram F-score (0..1), good enough to compare translations
    def grams(s, k): s = re.sub(r"\s+", " ", s.strip()); return Counter(s[i:i+k] for i in range(max(0, len(s)-k+1)))
    ps, rs = [], []
    for k in range(1, n+1):
        h, r = grams(hyp, k), grams(ref, k); m = sum((h & r).values())
        ps.append(m / max(1, sum(h.values()))); rs.append(m / max(1, sum(r.values())))
    p, r = sum(ps)/n, sum(rs)/n
    return 0 if p + r == 0 else 5*p*r/(4*p + r)

def jload(s):
    s = s.strip(); s = re.sub(r"^```(json)?|```$", "", s).strip()
    try: return json.loads(s[s.index("{"): s.rindex("}")+1])
    except Exception: return None

def score(task, out, ref):
    if task in ("code", "und"):
        m = re.match(r"\s*PLAN:\s*(edit|new)", out); rp = re.match(r"PLAN:\s*(edit|new)", ref).group(1)
        a, b = out.find("<xml"), out.rfind("</xml>")
        if a < 0 or b < a: return 0.0
        xml = out[a:b+6]
        types = set(re.findall(r'<(?:block|shadow) type="([^"]+)"', xml))
        if not types or not types <= catalog() or xml.count('type="page_root"') != 1: return 0.25
        names = set(re.findall(r'<field name="NAME">([^<]*)</field>', xml))
        if any(n not in names for n in re.findall(r'<field name="EL">([^<]*)</field>', xml)): return 0.5
        return 1.0 if (m and m.group(1) == rp) else 0.75
    if task == "tr": return chrf(out, ref)
    j, r = jload(out), json.loads(ref)
    if not j: return 0.0
    if task == "formula":
        f = lambda s: re.sub(r"\s+", "", str(s or "")).upper()
        return 1.0 if f(j.get("formula")) == f(r["formula"]) else 0.5 if str(j.get("formula", "")).startswith("=") else 0.25
    if task == "table":
        rows = j.get("rows")
        if not isinstance(rows, list) or not rows or not all(isinstance(x, list) for x in rows): return 0.25
        w = len(rows[0]); ok = all(len(x) == w for x in rows)
        return (0.5 if ok else 0.25) + (0.25 if w == len(r["rows"][0]) else 0) + (0.25 if len(rows) == len(r["rows"]) else 0)
    if task == "graph":
        return 0.4*(j.get("type") == r["type"]) + 0.3*(j.get("series") == r["series"]) + 0.3*(j.get("range") == r["range"])
    return 0.0

random.seed(7)
EVAL = [r for t in ["code", "und", "tr", "graph", "formula", "table"] for r in random.sample([x for x in test_rows if x["task"] == t], 4)]

def run_eval(label):
    FastLanguageModel.for_inference(model)
    per = {}
    for i, r in enumerate(EVAL):
        msgs = r["messages"]; ids = tokenizer(prompt_text(msgs), return_tensors="pt").to("cuda")
        long = r["task"] in ("code", "und")
        out = model.generate(**ids, max_new_tokens=1800 if long else 300, do_sample=False, temperature=None, top_p=None, top_k=None)
        text = tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
        per.setdefault(r["task"], []).append(score(r["task"], text, msgs[2]["content"]))
        print(f"\r{label}: {i+1}/{len(EVAL)}", end="")
    res = {t: round(100*sum(v)/len(v)) for t, v in per.items()}
    print("\n" + label, res, "평균", round(sum(res.values())/len(res)))
    return res

before = run_eval("학습 전") if RUN_EVAL else None

## 4. 학습

In [ ]:
model = FastLanguageModel.get_peft_model(model, r=CFG["r"], lora_alpha=CFG["r"]*2, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)

from datasets import Dataset
texts = [full_text(r["messages"]) for r in train_rows]
keep = [t for t in texts if len(tokenizer(t)["input_ids"]) <= MAX_LEN]
print(f"학습 자료 {len(keep)}개 (너무 긴 {len(texts)-len(keep)}개 제외)")
ds = Dataset.from_dict({"text": keep})

# Save progress to Google Drive every 100 steps so a cut-off runtime (usage limit, disconnect) can resume instead of starting over.
import os, glob
CKPT = f"/kaggle/working/ff-ckpt-{MODEL}"
import shutil
for old in glob.glob(f"/kaggle/input/**/ff-ckpt-{MODEL}", recursive=True):   # an earlier cut-off run added as Input
    if not glob.glob(os.path.join(CKPT, "checkpoint-*")) and glob.glob(os.path.join(old, "checkpoint-*")):
        shutil.copytree(old, CKPT, dirs_exist_ok=True); print("이전 진행 상황을 가져왔어요:", old)
print("진행 상황을 여기에 저장해요:", CKPT)

from trl import SFTTrainer, SFTConfig
args = dict(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=CFG["epochs"], learning_rate=CFG["lr"], warmup_ratio=0.03, lr_scheduler_type="cosine",
    logging_steps=10, optim="adamw_8bit", weight_decay=0.01, seed=3407, output_dir=CKPT, report_to="none",
    save_strategy="steps", save_steps=100, save_total_limit=1, fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported())
try: cfg = SFTConfig(max_length=MAX_LEN, **args)
except TypeError: cfg = SFTConfig(max_seq_length=MAX_LEN, **args)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=cfg)

from unsloth.chat_templates import train_on_responses_only   # learn only the answers, not the prompts
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
resume = bool(glob.glob(os.path.join(CKPT, "checkpoint-*")))
if resume: print("저장된 중간 진행에서 이어서 해요")
t0 = time.time(); trainer.train(resume_from_checkpoint=resume or None); print(f"학습 끝: {(time.time()-t0)/60:.0f}분")
import shutil; shutil.rmtree(CKPT, ignore_errors=True)   # finished: drop the checkpoint so the next run starts fresh

## 5. 시험 채점 (학습 후)

In [ ]:
after = run_eval("학습 후") if RUN_EVAL else None
if before and after:
    names = {"code": "블록 코딩", "und": "요청 이해", "tr": "번역", "graph": "그래프", "formula": "수식", "table": "표"}
    for t in names: print(f"{names[t]:6s} {before.get(t,0):3d} → {after.get(t,0):3d}")

## 6. 학습한 모델 올리기

학습한 모델을 Hugging Face에 비공개로 올려요. 브라우저용 변환은 GitHub(tools 저장소의 Convert model for the browser)에서 해요.

In [ ]:
import json, os
import shutil
shutil.rmtree("merged", ignore_errors=True)   # never upload an older model left over in this session
model.save_pretrained_merged("merged", tokenizer, save_method="merged_16bit")
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN); user = api.whoami()["name"]
rid = f"{user}/FF-Tools-{CFG['size']}-merged"
api.create_repo(rid, private=True, exist_ok=True)
api.upload_folder(folder_path="merged", repo_id=rid, commit_message="Tools fine-tune")
print("올림 끝:", rid)
print("이제 Claude에게 알려 주세요. 브라우저용 변환은 GitHub에서 해요.")